# Master Thesis VECM Pipeline

This is the parent runner for the thesis model rebuild. The econometric logic lives in `src/`; this notebook orchestrates the full workflow in a transparent, thesis-readable sequence.

Default behavior is strict gatekeeping: if stationarity, cointegration, adjustment significance, or diagnostics fail, the notebook records the failure instead of silently continuing.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/thesis_model_mpl")
os.environ.setdefault("MPLBACKEND", "Agg")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import Markdown, display

from src.cointegration import (
    cointegration_gate,
    johansen_rank_table,
    lag_selection_table,
    rank_sensitivity_table,
    selected_rank,
)
from src.config import (
    FIGURES_DIR,
    LEGACY_DATASET,
    REPORTS_DIR,
    TABLES_DIR,
    SampleConfig,
    ensure_output_dirs,
)
from src.data_processing import (
    data_integrity_report,
    load_legacy_raw_data,
    load_legacy_shocks,
    transform_legacy_data,
    variable_manifest,
)
from src.diagnostics import (
    companion_eigenvalues,
    covid_break_screen,
    diagnostics_gate,
    model_level_tests,
    residual_diagnostics,
)
from src.irf import (
    first_stage_table,
    proxy_first_stage,
    proxy_impact_vector,
    proxy_irf_table,
    standard_irf_table,
)
from src.plotting import plot_proxy_irfs, plot_time_series
from src.stationarity import stationarity_gate, stationarity_table
from src.vecm_model import alpha_table, beta_table, fit_vecm, short_run_table, significance_gate

ensure_output_dirs()
display(Markdown(f"**Project root:** `{PROJECT_ROOT}`"))

## Configuration

Set `ALLOW_EXPERIMENTAL = True` only when you deliberately want to run the full legacy U.S./Fed system for code validation. Thesis-grade inference should remain strict.

In [ ]:
DATASET = "legacy_us"
ALLOW_EXPERIMENTAL = False

SAMPLE = SampleConfig(
    start="2005-01-01",
    end="2025-12-31",
    covid_split="2020-03-31",
)

MAXLAGS = 12
LAG_CRITERION = "hqic"
RANK_METHOD = "trace"
DET_ORDER = 0
DETERMINISTIC = "co"
IRF_PERIODS = 36
DIAGNOSTIC_LAGS = 12

PIPELINE_MESSAGES: list[str] = []
STOP_REASON: str | None = None
selected_lag: int | None = None
rank: int | None = None
ok_diag: bool | None = None

display(Markdown(
    f"**Dataset:** `{DATASET}`  \n"
    f"**Strict mode:** `{not ALLOW_EXPERIMENTAL}`  \n"
    f"**Sample:** `{SAMPLE.start}` to `{SAMPLE.end}`"
))

## Step 1 — Data Cleaning And Transformation

In [ ]:
if DATASET != "legacy_us":
    raise ValueError("Only the local legacy_us dataset is currently available in the workspace.")

raw = load_legacy_raw_data(SAMPLE)
data = transform_legacy_data(raw)
shocks = load_legacy_shocks(SAMPLE)
variables = list(data.columns)

raw.to_csv(TABLES_DIR / "legacy_raw_aligned.csv")
data.to_csv(TABLES_DIR / "legacy_transformed_vecm_input.csv")
shocks.to_csv(TABLES_DIR / "legacy_external_shocks.csv")

integrity = data_integrity_report(raw, data)
manifest = variable_manifest()
integrity.to_csv(TABLES_DIR / "data_integrity.csv", index=False)
manifest.to_csv(TABLES_DIR / "variable_manifest.csv", index=False)

plot_time_series(data, FIGURES_DIR / "legacy_transformed_levels.png", "Legacy transformed VECM inputs")

display(Markdown("### Variable Manifest"))
display(manifest)
display(Markdown("### Data Integrity"))
display(integrity)
display(Markdown(f"Saved level plot to `{FIGURES_DIR / 'legacy_transformed_levels.png'}`."))

## Step 2 — Stationarity Testing

In [ ]:
stationarity = stationarity_table(data)
stationarity.to_csv(TABLES_DIR / "stationarity_tests.csv", index=False)

ok_stationarity, stationarity_message = stationarity_gate(stationarity)
PIPELINE_MESSAGES.append(stationarity_message)

display(stationarity)

if not ok_stationarity and not ALLOW_EXPERIMENTAL:
    STOP_REASON = "STOPPED_BY_STATIONARITY_GATE"
    display(Markdown(f"### Pipeline stopped: stationarity gate\n{stationarity_message}"))
else:
    display(Markdown(f"### Stationarity gate\n{stationarity_message}"))

## Steps 3 And 4 — Lag Selection And Cointegration Testing

In [ ]:
if STOP_REASON:
    display(Markdown(f"Skipped because pipeline status is `{STOP_REASON}`."))
else:
    lag_table, selected_orders = lag_selection_table(data, maxlags=MAXLAGS, deterministic=DETERMINISTIC)
    lag_table.to_csv(TABLES_DIR / "lag_selection.csv", index=False)
    selected_lag = int(selected_orders.get(LAG_CRITERION, selected_orders.get("hqic", 1)))
    PIPELINE_MESSAGES.append(f"Lag selection used {LAG_CRITERION.upper()} and selected k_ar_diff={selected_lag}.")

    sensitivity = rank_sensitivity_table(
        data,
        lag_values=sorted(set([max(0, selected_lag - 1), selected_lag, selected_lag + 1])),
        det_orders=[-1, 0, 1],
    )
    sensitivity.to_csv(TABLES_DIR / "cointegration_rank_sensitivity.csv", index=False)

    rank = selected_rank(data, selected_lag, DET_ORDER, method=RANK_METHOD)
    johansen = johansen_rank_table(data, selected_lag, DET_ORDER)
    johansen.to_csv(TABLES_DIR / "johansen_rank_test.csv", index=False)

    ok_coint, coint_message = cointegration_gate(rank, data.shape[1])
    PIPELINE_MESSAGES.append(coint_message)

    display(Markdown("### Lag Selection"))
    display(lag_table)
    display(Markdown("### Rank Sensitivity"))
    display(sensitivity)
    display(Markdown("### Johansen Test"))
    display(johansen)

    if not ok_coint and not ALLOW_EXPERIMENTAL:
        STOP_REASON = "STOPPED_BY_COINTEGRATION_GATE"
        display(Markdown(f"### Pipeline stopped: cointegration gate\n{coint_message}"))
    else:
        display(Markdown(f"### Cointegration gate\n{coint_message}"))

## Step 5 — Estimate VECM And Interpret Adjustment

In [ ]:
if STOP_REASON:
    display(Markdown(f"Skipped because pipeline status is `{STOP_REASON}`."))
else:
    result = fit_vecm(data, k_ar_diff=selected_lag, coint_rank=rank, deterministic=DETERMINISTIC)
    (REPORTS_DIR / "vecm_summary.txt").write_text(str(result.summary()), encoding="utf-8")

    alpha = alpha_table(result, variables)
    beta = beta_table(result, variables)
    short_run = short_run_table(result, variables)

    alpha.to_csv(TABLES_DIR / "alpha_adjustment.csv", index=False)
    beta.to_csv(TABLES_DIR / "beta_cointegration_vectors.csv", index=False)
    short_run.to_csv(TABLES_DIR / "short_run_coefficients.csv", index=False)

    ok_alpha, alpha_message = significance_gate(alpha)
    PIPELINE_MESSAGES.append(alpha_message)

    display(Markdown("### Alpha Adjustment"))
    display(alpha)
    display(Markdown("### Beta Cointegration Vectors"))
    display(beta)

    if not ok_alpha and not ALLOW_EXPERIMENTAL:
        STOP_REASON = "STOPPED_BY_ADJUSTMENT_GATE"
        display(Markdown(f"### Pipeline stopped: adjustment gate\n{alpha_message}"))
    else:
        display(Markdown(f"### Adjustment gate\n{alpha_message}"))

## Steps 6, 7 And 8 — Diagnostics, Stability, Break Screen, And IRFs

In [ ]:
if STOP_REASON:
    display(Markdown(f"Skipped because pipeline status is `{STOP_REASON}`."))
else:
    resid_diag = residual_diagnostics(result, data.index, variables, lags=DIAGNOSTIC_LAGS)
    model_tests = model_level_tests(result)
    eig = companion_eigenvalues(result)
    breaks = covid_break_screen(data, split_date=SAMPLE.covid_split)

    resid_diag.to_csv(TABLES_DIR / "residual_diagnostics.csv", index=False)
    model_tests.to_csv(TABLES_DIR / "model_level_diagnostics.csv", index=False)
    eig.to_csv(TABLES_DIR / "companion_eigenvalues.csv", index=False)
    breaks.to_csv(TABLES_DIR / "covid_break_screen.csv", index=False)

    ok_diag, diag_message = diagnostics_gate(resid_diag)
    PIPELINE_MESSAGES.append(diag_message)

    standard_irf_table(result, variables, periods=IRF_PERIODS).to_csv(TABLES_DIR / "standard_irfs.csv", index=False)

    if LEGACY_DATASET.policy_column in variables and LEGACY_DATASET.shock_column in shocks.columns:
        instrument, residuals, _, first_stage = proxy_first_stage(
            result,
            data.index,
            variables,
            shocks,
            policy_column=LEGACY_DATASET.policy_column,
            shock_column=LEGACY_DATASET.shock_column,
        )
        first_stage_table(first_stage).to_csv(TABLES_DIR / "proxy_first_stage.csv", index=False)
        impact = proxy_impact_vector(residuals, instrument, policy_column=LEGACY_DATASET.policy_column)
        impact.to_frame().to_csv(TABLES_DIR / "proxy_impact_vector.csv")
        proxy_irfs = proxy_irf_table(result, impact, periods=IRF_PERIODS)
        proxy_irfs.to_csv(TABLES_DIR / "proxy_irfs.csv", index=False)
        plot_proxy_irfs(proxy_irfs, FIGURES_DIR / "legacy_proxy_irfs.png", "Legacy proxy-VECM IRFs")
        PIPELINE_MESSAGES.append("Proxy IRFs generated with the local Fed JK monetary policy shock file.")
    else:
        PIPELINE_MESSAGES.append("Proxy IRFs skipped because policy column or shock column is unavailable.")

    display(Markdown("### Residual Diagnostics"))
    display(resid_diag)
    display(Markdown("### COVID Break Screen"))
    display(breaks)
    display(Markdown(f"### Diagnostics gate\n{diag_message}"))

## Final Pipeline Report

In [ ]:
def write_pipeline_report() -> str:
    if STOP_REASON:
        status = STOP_REASON
    else:
        status = "COMPLETED_EXPERIMENTAL" if ALLOW_EXPERIMENTAL and not ok_stationarity else "COMPLETED"
        if ok_diag is False:
            status += "_WITH_DIAGNOSTIC_WARNINGS"

    lines = [
        "# Thesis VECM Pipeline Report",
        "",
        f"Status: {status}",
        f"Experimental override: {ALLOW_EXPERIMENTAL}",
        "",
        "## Gatekeeping Messages",
        "",
    ]
    lines.extend([f"- {message}" for message in PIPELINE_MESSAGES])
    if selected_lag is not None:
        lines.extend(["", f"Selected short-run lag order: {selected_lag}"])
    if rank is not None:
        lines.extend(["", f"Selected cointegration rank: {rank}"])
    lines.extend(
        [
            "",
            "## Research Interpretation",
            "",
            "The current local files reproduce the old U.S./Fed-oriented workflow. They are useful for code validation and audit, but they do not satisfy the thesis requirement for a Germany-first ECB/Bundesbank/Eurostat data system.",
            "",
            "A thesis-grade run should replace this legacy dataset with Germany/Euro Area series before accepting VECM estimates or impulse responses as evidence for the thesis.",
        ]
    )

    report_name = "pipeline_report_experimental.md" if ALLOW_EXPERIMENTAL else "pipeline_report.md"
    report_path = REPORTS_DIR / report_name
    report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return status

final_status = write_pipeline_report()
display(Markdown(f"## Final status: `{final_status}`"))
display(Markdown(f"Reports saved in `{REPORTS_DIR}`."))